# Ridge and Lasso Regression

**Topic:** Supervised Learning — Regression

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how Ridge and Lasso add a penalty term to the linear regression loss function
- **Explain** the key difference between them: Ridge shrinks coefficients toward zero, Lasso drives some to exactly zero
- **Interpret** a coefficient path plot showing how coefficient values change as regularization strength increases

> **Tip:** In the **Coefficient Shrinkage Explorer**, toggle between Ridge and Lasso — each has its own alpha slider, and notice the sliders cover very different ranges. Lasso's interesting range is roughly 0.001–2; Ridge needs to climb into the thousands or higher before you see comparable shrinkage. That difference in scale is itself part of the lesson.

---
## How we got here

Regularization is the answer to the overfitting problem you saw in polynomial regression:

- **[supervised/03_polynomial_regression.ipynb](03_polynomial_regression.ipynb)** — the overfitting problem and why unconstrained MSE minimization goes wrong at high complexity
- **[math/statistics_for_ml/04_regularization_mathematics.ipynb](../math/statistics_for_ml/04_regularization_mathematics.ipynb)** — the mathematical derivation of why adding a penalty to the loss function shrinks coefficients
- **[ml_concepts/04_bias_variance_tradeoff.ipynb](../ml_concepts/04_bias_variance_tradeoff.ipynb)** — regularization is a deliberate way to increase bias slightly in exchange for a larger reduction in variance

---
## Why this matters for data science

Ridge and Lasso are the most widely used regularized linear models in practice. They solve two of the biggest problems with plain linear regression: **overfitting to noise and unstable coefficients when features are correlated.**

Lasso has a unique and extremely valuable property: it performs **automatic feature selection** by setting irrelevant coefficients to exactly zero. When you have hundreds of candidate features and only some of them matter, Lasso finds the sparse subset for you. Ridge is better when all features are relevant and you just want to reduce their magnitudes. Elastic Net combines both.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Ridge and Lasso sit at the **same top-left position** as linear regression, but they are more robust to high-dimensional or correlated feature spaces. The penalty shrinks the model's effective complexity even when you have many features, keeping interpretability intact.

As alpha increases (stronger regularization), the model moves slightly down and to the left: it becomes even more constrained, simpler, and more biased — but also less prone to variance. At very high alpha, most coefficients approach zero and the model becomes nearly a constant.

---
## How it learns

Think about what happens when you fit linear regression on noisy data with many features. The algorithm will happily assign a large coefficient to a feature that by chance correlates with the noise in the training set. That large coefficient is not real — it is just overfitting.

Regularization fixes this by taxing large coefficients. The model now minimizes not just prediction error but also the size of its own weights. To earn a large coefficient, a feature has to produce a big enough reduction in MSE to pay the penalty. Features that only correlate with noise do not make that trade — their coefficients shrink or disappear.

Ridge and Lasso use different penalties. Ridge charges proportional to the **squared** size of each coefficient — it shrinks everything proportionally. Lasso charges proportional to the **absolute** size — it has a corner in the cost function geometry that produces exact zeros.

---
## The math behind it

Both methods add a **regularization term** to the MSE loss:

**Ridge (L2 regularization):**

$$J_{\text{Ridge}}(\boldsymbol{\theta}) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2 + \alpha \sum_{j=1}^{p} \theta_j^2$$

**Lasso (L1 regularization):**

$$J_{\text{Lasso}}(\boldsymbol{\theta}) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2 + \alpha \sum_{j=1}^{p} |\theta_j|$$

- $\alpha$ — regularization strength (hyperparameter); $\alpha = 0$ gives plain linear regression
- The penalty applies only to feature coefficients $\theta_1, \ldots, \theta_p$, not the intercept $\theta_0$

**Why Lasso produces zeros:** the L1 penalty has a non-differentiable point at $\theta_j = 0$. The subdifferential of the absolute value function includes zero, so the optimization can land exactly at zero and stay there. Ridge uses a smooth quadratic penalty that approaches but never reaches zero.

**Optimization criterion:** minimize the penalized MSE. Ridge has a closed-form solution; Lasso uses coordinate descent because the L1 penalty is not differentiable at zero.

---
## Try it yourself

Two widgets below: the **Coefficient Shrinkage Explorer** shows all eight coefficients as regularization strength increases, and the **Regularization Boundary Widget** shows what a Ridge/Lasso fit actually looks like against the data as alpha changes.

### Coefficient Shrinkage Explorer

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing
import numpy as np

np.random.seed(42)
_h_w1 = fetch_california_housing(as_frame=True)
_X_w1 = StandardScaler().fit_transform(_h_w1.data)
_y_w1 = _h_w1.target.values
_feat_w1 = list(_h_w1.feature_names)

out_shrink = Output()

model_tg_w1 = widgets.ToggleButtons(options=["Ridge", "Lasso"], value="Ridge")
# Ridge and Lasso need very different alpha ranges to show comparable shrinkage
# on this dataset (~20k rows, 8 features) - one shared slider can't serve both,
# so each model gets its own, and only the active one is enabled.
ridge_alpha_sl_w1 = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=6, step=0.1,
    description="Ridge alpha:", style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"))
lasso_alpha_sl_w1 = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0.5, step=0.05,
    description="Lasso alpha:", style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"), disabled=True)

def render_shrink(change=None):
    if model_tg_w1.value == "Ridge":
        alpha = ridge_alpha_sl_w1.value
        model = Ridge(alpha=alpha)
    else:
        alpha = lasso_alpha_sl_w1.value
        model = Lasso(alpha=alpha, max_iter=5000)
    model.fit(_X_w1, _y_w1)
    n_zero = int(np.sum(np.isclose(model.coef_, 0.0, atol=1e-6)))
    colors = [PALETTE["muted"] if np.isclose(c, 0.0, atol=1e-6)
              else (PALETTE["primary"] if c > 0 else PALETTE["secondary"])
              for c in model.coef_]
    fig = go.Figure(data=go.Bar(
        x=_feat_w1, y=model.coef_, marker_color=colors,
        text=[f"{c:+.3f}" for c in model.coef_], textposition="outside",
    ), layout=base_layout(
        title=f"{model_tg_w1.value} coefficients at α={alpha:.4f}  |  {n_zero} feature(s) zeroed out",
        xaxis_title="Feature", yaxis_title="Coefficient",
    ))
    fig.update_layout(showlegend=False)
    with out_shrink:
        clear_output(wait=True)
        fig.show()

def _sync_sliders_w1(change=None):
    is_ridge = model_tg_w1.value == "Ridge"
    ridge_alpha_sl_w1.disabled = not is_ridge
    lasso_alpha_sl_w1.disabled = is_ridge
    render_shrink()

model_tg_w1.observe(_sync_sliders_w1, names="value")
ridge_alpha_sl_w1.observe(render_shrink, names="value")
lasso_alpha_sl_w1.observe(render_shrink, names="value")
display(VBox([model_tg_w1, ridge_alpha_sl_w1, lasso_alpha_sl_w1, out_shrink]))
render_shrink()


### Regularization Boundary Widget

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.datasets import fetch_california_housing
import numpy as np

np.random.seed(42)
_h_w2 = fetch_california_housing(as_frame=True)
_X_w2 = StandardScaler().fit_transform(_h_w2.data)
_y_w2 = _h_w2.target.values
_medinc_idx_w2 = list(_h_w2.feature_names).index("MedInc")
_idx_sample_w2 = np.random.RandomState(42).choice(len(_y_w2), 500, replace=False)

out_reg_fit = Output()

model_tg_w2 = widgets.ToggleButtons(options=["Ridge", "Lasso"], value="Ridge")
ridge_alpha_sl_w2 = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=6, step=0.1,
    description="Ridge alpha:", style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"))
lasso_alpha_sl_w2 = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0.5, step=0.05,
    description="Lasso alpha:", style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"), disabled=True)

def render_reg_fit(change=None):
    if model_tg_w2.value == "Ridge":
        alpha = ridge_alpha_sl_w2.value
        model = Ridge(alpha=alpha)
    else:
        alpha = lasso_alpha_sl_w2.value
        model = Lasso(alpha=alpha, max_iter=5000)
    model.fit(_X_w2, _y_w2)
    y_pred_all = model.predict(_X_w2)
    r2 = r2_score(_y_w2, y_pred_all)
    rmse = np.sqrt(mean_squared_error(_y_w2, y_pred_all))
    n_nonzero = int(np.sum(~np.isclose(model.coef_, 0.0, atol=1e-6)))

    X_sample = _X_w2[_idx_sample_w2]
    y_sample = _y_w2[_idx_sample_w2]
    x_medinc_sample = X_sample[:, _medinc_idx_w2]

    # Genuine partial-dependence curve: every feature is standardized (mean 0),
    # so holding "everything else at its mean" means setting every other
    # coordinate to exactly 0 and sweeping MedInc alone. This replaces the old
    # approach of connecting real predictions in MedInc order, which produced a
    # jagged zigzag because the other 7 features varied freely across points.
    medinc_grid = np.linspace(x_medinc_sample.min(), x_medinc_sample.max(), 200)
    X_curve = np.zeros((len(medinc_grid), _X_w2.shape[1]))
    X_curve[:, _medinc_idx_w2] = medinc_grid
    y_curve = model.predict(X_curve)

    fig = go.Figure(data=[
        go.Scatter(x=x_medinc_sample, y=y_sample, mode="markers",
            marker=dict(color=PALETTE["muted"], size=5, opacity=0.4), name="Districts"),
        go.Scatter(x=medinc_grid, y=y_curve, mode="lines",
            line=dict(color=PALETTE["primary"], width=3),
            name=f"{model_tg_w2.value} fit (other features at their mean)"),
    ], layout=base_layout(
        title=f"{model_tg_w2.value} α={alpha:.4f}  |  R²={r2:.3f}  RMSE={rmse:.3f}  |  {n_nonzero}/8 nonzero coefs",
        xaxis_title="MedInc (standardized)", yaxis_title="Median House Value",
    ))
    with out_reg_fit:
        clear_output(wait=True)
        fig.show()

def _sync_sliders_w2(change=None):
    is_ridge = model_tg_w2.value == "Ridge"
    ridge_alpha_sl_w2.disabled = not is_ridge
    lasso_alpha_sl_w2.disabled = is_ridge
    render_reg_fit()

model_tg_w2.observe(_sync_sliders_w2, names="value")
ridge_alpha_sl_w2.observe(render_reg_fit, names="value")
lasso_alpha_sl_w2.observe(render_reg_fit, names="value")
display(VBox([model_tg_w2, ridge_alpha_sl_w2, lasso_alpha_sl_w2, out_reg_fit]))
render_reg_fit()


---
## What's happening?

In the **Coefficient Shrinkage Explorer**, alpha controls how heavy the tax on large coefficients is. Notice the two sliders don't share a range: Lasso reaches full sparsity (every coefficient zeroed) by around alpha ≈ 1, while Ridge is still barely shrunk at alpha = 1,000 and only approaches the mean prediction somewhere past alpha = 100,000. That's not an accident of this widget — it's a real property of the two penalties on real-scale data, and it's why you should never assume the same alpha value means the same thing for L1 vs. L2 regularization.

The difference between Ridge and Lasso is geometric. Ridge's L2 penalty draws a sphere around the origin in coefficient space. Lasso's L1 penalty draws a diamond (L1 ball). The optimal solution lands where the MSE contours first touch the constraint boundary. Sphere corners are rare — solutions land on the smooth surface and coefficients shrink proportionally. Diamond corners are at the axes — solutions often land exactly at a corner, giving a coefficient of zero.

In the **Regularization Boundary Widget**, the curve shown is a partial-dependence line — it holds every other feature at its mean and shows only how the prediction responds to MedInc. As alpha increases, that curve flattens toward a constant: maximum bias, minimum variance. The best alpha is somewhere in between, and cross-validation finds it.

---
## Key hyperparameters

**`alpha`** (default `1.0`) — regularization strength. Larger values shrink coefficients more aggressively. This is the single most important hyperparameter. Always tune it with cross-validation (RidgeCV or LassoCV).

**`max_iter`** (default `1000` for Lasso) — maximum number of coordinate descent iterations. Increase this if Lasso does not converge on your dataset. Ridge does not need this (it has a closed-form solution).

**`tol`** (default `1e-4`) — convergence tolerance for iterative solvers (Lasso). Lower values give more precise solutions but take longer.

For the full list of hyperparameters, see the sklearn documentation:
[Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) |
[Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Solves multicollinearity and overfitting in one step | Adds a hyperparameter (alpha) that must be tuned |
| Lasso does automatic feature selection (exact zeros) | Lasso can arbitrarily pick one of several correlated features and zero out the rest |
| Ridge is numerically stable even when p > n | Very high alpha forces underfitting |
| Fast to train and interpretable (still linear coefficients) | Standard errors and p-values require extra work (use statsmodels for inference) |
| Elastic Net combines both penalties for the best of both | Scale-sensitive: features must be standardized before regularization means anything |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| You have many features and suspect some are irrelevant (Lasso) | You need individual coefficient significance (use plain linear regression + statsmodels) |
| Features are correlated and plain linear regression is unstable (Ridge) | Your features are already sparse or you know all are important (regularization adds unnecessary bias) |
| You have more features than training examples (p > n) | You are using count or text data with nonnegative constraints (try Ridge with `positive=True`) |
| You want implicit feature selection without an extra selection step (Lasso) | The nonlinearity in the data is large (regularized linear still underfits) |
| You are fitting polynomial features and worry about overfitting | You need online/incremental learning (use SGDRegressor with L1 or L2 penalty) |

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error
import numpy as np, plotly.graph_objects as go

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
_X_all = housing.data
_y_all = housing.target.values

# On the full ~20k-row dataset, all 8 real features are genuinely informative,
# so cross-validation correctly picks a tiny alpha for Lasso and nothing gets
# zeroed - there's nothing irrelevant to prune. A smaller subsample plus known-
# irrelevant synthetic noise features gives Lasso something legitimate to zero
# out, so the feature-selection story below is real and reproducible.
_idx = np.random.RandomState(42).choice(len(_X_all), 400, replace=False)
X = _X_all.iloc[_idx].reset_index(drop=True).copy()
y = _y_all[_idx]

_rng = np.random.RandomState(42)
n_noise = 12
for i in range(n_noise):
    X[f"Noise_{i+1}"] = _rng.normal(0, 1, len(X))

features = list(X.columns)
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

alphas = np.logspace(-3, 3, 100)
ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
lasso = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42).fit(X_train, y_train)

zeroed_by_lasso = [f for f, c in zip(features, lasso.coef_) if np.isclose(c, 0, atol=1e-6)]

for name, model in [("Ridge", ridge), ("Lasso", lasso)]:
    y_pred = model.predict(X_test)
    nonzero = sum(c != 0 for c in model.coef_)
    print(f"{name:6s}  alpha={model.alpha_:.4f}  "
          f"R²={r2_score(y_test, y_pred):.4f}  "
          f"RMSE={root_mean_squared_error(y_test, y_pred):.4f}  "
          f"nonzero_coefs={nonzero}/{len(features)}")
print(f"\nFeatures zeroed by Lasso: {zeroed_by_lasso}")

fig = go.Figure(data=[
    go.Bar(name=f"Ridge (α={ridge.alpha_:.3f})",
           x=features, y=ridge.coef_,
           marker_color=PALETTE["primary"], opacity=0.8),
    go.Bar(name=f"Lasso (α={lasso.alpha_:.4f})",
           x=features, y=lasso.coef_,
           marker_color=PALETTE["secondary"], opacity=0.8),
], layout=base_layout(
    title="Ridge vs Lasso — Standardized Coefficients (8 real features + 12 synthetic noise features)",
    xaxis_title="Feature",
    yaxis_title="Coefficient",
))
fig.update_layout(barmode="group")
fig.update_xaxes(tickangle=-45)
fig.show()


---
## Real-world example: Ridge vs. Lasso coefficient comparison

The chart above uses the 8 real California Housing features plus 12 synthetic noise features (randomly generated, with no real relationship to price) on a 400-row subsample. Ridge and Lasso are both tuned by 5-fold cross-validation, and the difference in what they do with the noise features is the whole point of this example.

- **Notice:** Ridge keeps all 20 features nonzero — including all 12 noise columns — it down-weights weak signals but never eliminates them
- **Notice:** Lasso zeros out several of the noise features, plus `AveRooms` — a real feature, but one that's highly correlated with `AveBedrms` (see the multicollinearity discussion in notebook 02). This is the exact "arbitrarily drops one of several correlated features" behavior mentioned in the strengths/weaknesses table above, caught in the act
- **Notice:** Lasso achieves comparable or better test R² than Ridge here, with meaningfully fewer active features — a real benefit for interpretability and deployment, not just a theoretical one

> **Discussion question:** In a medical risk model, would you prefer Ridge or Lasso? Does the answer change if doctors need to understand why the model made a specific prediction?

### Regularization methods comparison

| Method | Penalty | Feature selection | Best for |
|---|---|---|---|
| Ridge (L2) | Sum of squared coefs | No: all features kept | Correlated features, stable coefficients |
| Lasso (L1) | Sum of absolute coefs | Yes: exact zeros | Many irrelevant features, sparse models |
| Elastic Net | Ridge + Lasso combined | Partial | Both correlated features and irrelevant ones |
| Plain Linear | None | No | When you want maximum statistical power |

> **Ridge shrinks all coefficients toward zero proportionally; Lasso drives some to exactly zero, making it the only linear model that performs automatic feature selection.**

---
*Next up: 05 — Logistic Regression, the classification counterpart that turns a linear score into a probability*